# 07. Entrenamiento de Modelos Híbridos (RandomForest / XGBoost)

**Objetivo:** entrenar y comparar clasificadores con las features híbridas exportadas por 06.
**Entradas:** `features_core.parquet`, `features_py.parquet`, `data/splits/*_indices.csv`.
**Salidas:** `data/outputs/train_<run_id>/comparacion_modelos_<split>.csv`, `predicciones_*_<split>.csv`, `figures/*.png`, `modelo_*.joblib`, `resumen_entrenamiento.json`.
**Notebook anterior:** `notebooks/pipeline/06_ingenieria_features_hibridas.ipynb`.
**Notebook siguiente:** `notebooks/pipeline/08_resultados_hibrido_vs_lineas_base.ipynb` y `notebooks/analysis/09_analisis_errores_hibrido.ipynb`.


## Criterio metodológico
- Comparación `core` vs `py` como ablación controlada.
- Comparación de clasificadores: `RandomForest` vs `XGBoost`.
- Se permite `RandomizedSearchCV` opcional para tuning.
- Split de evaluación configurable (`dev` para ajuste, `test` para reporte final).


In [1]:
import json
import os
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.preprocessing import LabelEncoder

from utils_shared import setup_paths, load_splits, ensure_dir, guess_label_col, guess_patient_id_col, guess_text_col

warnings.filterwarnings('ignore')

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception as e:
    HAS_XGB = False
    print('Aviso: xgboost no disponible:', e)

paths = setup_paths()
BASE_PATH = paths['BASE_PATH']
DATA_PATH = paths['DATA_PATH']
SPLITS_PATH = paths['SPLITS_PATH']
PROCESSED_PATH = paths['PROCESSED_PATH']
OUTPUTS_PATH = paths['OUTPUTS_PATH']

print('HAS_XGB:', HAS_XGB)


HAS_XGB: True


In [2]:
# Configuración de entrenamiento y evaluación
# Búsqueda de hiperparámetros opcional con `RandomizedSearchCV` para modelos tabulares.

EVAL_ON = os.getenv('TRAIN_EVAL_ON', 'dev').strip().lower()  # 'dev' o 'test'
if EVAL_ON not in {'dev', 'test'}:
    raise ValueError(f"TRAIN_EVAL_ON inválido: {EVAL_ON}. Use 'dev' o 'test'.")

FEATURE_RUN_ID_CORE = os.getenv('TRAIN_FEATURE_RUN_ID_CORE') or None
FEATURE_RUN_ID_PY = os.getenv('TRAIN_FEATURE_RUN_ID_PY') or None

USE_RANDOMIZED_SEARCH = os.getenv('TRAIN_USE_RANDOM_SEARCH', '1') == '1'
USE_XGB = os.getenv('TRAIN_USE_XGB', '1') == '1'
REQUIRE_XGB = os.getenv('TRAIN_REQUIRE_XGB', '0') == '1'

N_ITER_SEARCH = int(os.getenv('TRAIN_N_ITER_SEARCH', '25'))
CV_FOLDS = int(os.getenv('TRAIN_CV_FOLDS', '3'))
RANDOM_SEED = int(os.getenv('TRAIN_RANDOM_SEED', '42'))
N_JOBS = int(os.getenv('TRAIN_N_JOBS', '-1'))

RUN_ID = os.getenv('TRAIN_RUN_ID') or pd.Timestamp.now().strftime('train_%Y%m%d_%H%M%S')
OUT_DIR = ensure_dir(OUTPUTS_PATH / RUN_ID)
FIG_DIR = ensure_dir(OUT_DIR / 'figures')

print('OUT_DIR:', OUT_DIR)
print('EVAL_ON:', EVAL_ON)
print('USE_RANDOMIZED_SEARCH:', USE_RANDOMIZED_SEARCH)
print('USE_XGB:', USE_XGB)
print('REQUIRE_XGB:', REQUIRE_XGB)

if USE_XGB and REQUIRE_XGB and not HAS_XGB:
    raise ImportError('Se solicitó XGBoost (TRAIN_REQUIRE_XGB=1), pero no está disponible en el entorno.')


OUT_DIR: /Users/manuelnunez/Projects/psych-phenotyping-paraguay/data/outputs/train_20260310_093418
EVAL_ON: dev
USE_RANDOMIZED_SEARCH: True
USE_XGB: True
REQUIRE_XGB: False


In [3]:
def _latest_feature_run(suffix: str):
    cand = sorted(PROCESSED_PATH.glob(f'fe_*_{suffix}'))
    return cand[-1].name if cand else None

FEATURE_RUN_ID_CORE = FEATURE_RUN_ID_CORE or _latest_feature_run('core')
FEATURE_RUN_ID_PY = FEATURE_RUN_ID_PY or _latest_feature_run('py')

if FEATURE_RUN_ID_CORE is None or FEATURE_RUN_ID_PY is None:
    raise FileNotFoundError('No se detectaron corridas de features en data/processed/fe_*_core|py')

print('FEATURE_RUN_ID_CORE:', FEATURE_RUN_ID_CORE)
print('FEATURE_RUN_ID_PY  :', FEATURE_RUN_ID_PY)


def load_features(run_id: str, suffix: str) -> pd.DataFrame:
    fpath = PROCESSED_PATH / run_id / f'features_{suffix}.parquet'
    if not fpath.exists():
        raise FileNotFoundError(f'No existe: {fpath}')
    return pd.read_parquet(fpath)


df_core = load_features(FEATURE_RUN_ID_CORE, 'core')
df_py = load_features(FEATURE_RUN_ID_PY, 'py')

print('core shape:', df_core.shape)
print('py   shape:', df_py.shape)

label_col = guess_label_col(df_py)
pid_col = guess_patient_id_col(df_py)
text_col = guess_text_col(df_py)
if label_col is None:
    raise ValueError('No se detectó columna de etiqueta en features.')

print('label_col:', label_col)
print('patient_id_col:', pid_col)
print('text_col:', text_col)


FEATURE_RUN_ID_CORE: fe_20260310_082139_core
FEATURE_RUN_ID_PY  : fe_20260310_082139_py
core shape: (1835, 958)
py   shape: (1835, 962)
label_col: etiqueta
patient_id_col: patient_id
text_col: texto


In [4]:
# Particiones patient-level
_, train_ids, dev_ids, test_ids = load_splits(SPLITS_PATH)
train_ids = set(train_ids)
dev_ids = set(dev_ids)
test_ids = set(test_ids)


def subset_by_split(df: pd.DataFrame, split: str) -> pd.DataFrame:
    idx = {'train': train_ids, 'dev': dev_ids, 'test': test_ids}[split]
    return df[df['row_id'].isin(idx)].copy()


def build_xy(df: pd.DataFrame):
    y = df[label_col].astype(str)
    meta = df[['row_id', label_col]].copy().rename(columns={label_col: 'y_true'})

    if pid_col is not None and pid_col in df.columns:
        grupos = df[pid_col].astype(str)
    else:
        grupos = df['row_id'].astype(str)

    drop_cols = {'row_id', label_col}
    if pid_col is not None and pid_col in df.columns:
        drop_cols.add(pid_col)
    if text_col in df.columns:
        drop_cols.add(text_col)

    X = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

    # Evita fallas por columnas no numéricas (ej. sent_label)
    non_num = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    if non_num:
        X = X.drop(columns=non_num)

    return X, y, meta, grupos


df_core_train = subset_by_split(df_core, 'train')
df_core_dev = subset_by_split(df_core, 'dev')
df_core_test = subset_by_split(df_core, 'test')

df_py_train = subset_by_split(df_py, 'train')
df_py_dev = subset_by_split(df_py, 'dev')
df_py_test = subset_by_split(df_py, 'test')

Xc_tr, yc_tr, Mc_tr, Gc_tr = build_xy(df_core_train)
Xc_dev, yc_dev, Mc_dev, _ = build_xy(df_core_dev)
Xc_te, yc_te, Mc_te, _ = build_xy(df_core_test)

Xp_tr, yp_tr, Mp_tr, Gp_tr = build_xy(df_py_train)
Xp_dev, yp_dev, Mp_dev, _ = build_xy(df_py_dev)
Xp_te, yp_te, Mp_te, _ = build_xy(df_py_test)

print('CORE X train:', Xc_tr.shape, '| y:', yc_tr.value_counts().to_dict())
print('PY   X train:', Xp_tr.shape, '| y:', yp_tr.value_counts().to_dict())


CORE X train: (1107, 954) | y: {'depresion': 749, 'ansiedad': 358}
PY   X train: (1107, 958) | y: {'depresion': 749, 'ansiedad': 358}


In [5]:
def _resolve_cv(groups):
    """Configura CV por grupos cuando hay suficiente diversidad de pacientes."""
    if groups is None:
        return CV_FOLDS, None

    n_groups = pd.Series(groups).nunique()
    if n_groups < 2:
        return CV_FOLDS, None

    n_splits = min(CV_FOLDS, n_groups)
    if n_splits < 2:
        return CV_FOLDS, None

    return GroupKFold(n_splits=n_splits), groups


def entrenar_rf(X_train, y_train, groups=None):
    base = RandomForestClassifier(
        n_estimators=500,
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS,
        class_weight='balanced',
    )

    if not USE_RANDOMIZED_SEARCH:
        base.fit(X_train, y_train)
        return base, {'modo': 'sin_tuning'}, np.nan

    cv, fit_groups = _resolve_cv(groups)

    param_dist = {
        'n_estimators': [300, 500, 800],
        'max_depth': [None, 8, 12, 16],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2', None],
    }
    search = RandomizedSearchCV(
        base,
        param_distributions=param_dist,
        n_iter=N_ITER_SEARCH,
        scoring='f1_macro',
        cv=cv,
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS,
        verbose=1,
    )
    if fit_groups is None:
        search.fit(X_train, y_train)
    else:
        search.fit(X_train, y_train, groups=fit_groups)
    return search.best_estimator_, search.best_params_, search.best_score_


def entrenar_xgb(X_train, y_train, groups=None):
    if not HAS_XGB:
        raise RuntimeError('xgboost no disponible')

    le = LabelEncoder()
    y_enc = le.fit_transform(pd.Series(y_train).astype(str))

    base = XGBClassifier(
        objective='multi:softprob',
        eval_metric='mlogloss',
        random_state=RANDOM_SEED,
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        tree_method='hist',
        num_class=len(le.classes_),
    )

    if not USE_RANDOMIZED_SEARCH:
        base.fit(X_train, y_enc)
        return base, {'modo': 'sin_tuning'}, np.nan, le

    cv, fit_groups = _resolve_cv(groups)

    param_dist = {
        'n_estimators': [200, 400, 700],
        'learning_rate': [0.03, 0.05, 0.1],
        'max_depth': [3, 5, 7, 9],
        'subsample': [0.7, 0.9, 1.0],
        'colsample_bytree': [0.7, 0.9, 1.0],
        'min_child_weight': [1, 3, 5],
        'reg_lambda': [0.5, 1.0, 2.0],
    }
    search = RandomizedSearchCV(
        base,
        param_distributions=param_dist,
        n_iter=N_ITER_SEARCH,
        scoring='f1_macro',
        cv=cv,
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS,
        verbose=1,
    )
    if fit_groups is None:
        search.fit(X_train, y_enc)
    else:
        search.fit(X_train, y_enc, groups=fit_groups)
    return search.best_estimator_, search.best_params_, search.best_score_, le


def evaluar_y_exportar(y_true, y_pred, titulo: str, prefijo: str):
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    bacc = balanced_accuracy_score(y_true, y_pred)
    labels = sorted(pd.Series(list(set(y_true) | set(y_pred))).unique())

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_norm = confusion_matrix(y_true, y_pred, labels=labels, normalize='true')

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm_norm)
    ax.set_xticks(range(len(labels)), labels=labels, rotation=45, ha='right')
    ax.set_yticks(range(len(labels)), labels=labels)
    ax.set_xlabel('Predicción')
    ax.set_ylabel('Real')
    ax.set_title(titulo)
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha='center', va='center')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'{prefijo}_confusion.png', dpi=200)
    plt.close(fig)

    report = classification_report(y_true, y_pred, digits=4, output_dict=True)
    pd.DataFrame(report).transpose().to_csv(OUT_DIR / f'reporte_{prefijo}.csv')

    return macro_f1, bacc


In [6]:
def ejecutar_experimentos(profile: str, X_tr, y_tr, G_tr, X_eval, y_eval, M_eval):
    filas = []
    modelos = {}

    # Comparación principal de clasificadores sobre las mismas características y particiones
    # RandomForest
    rf, rf_params, rf_cv = entrenar_rf(X_tr, y_tr, groups=G_tr)
    yhat_rf = rf.predict(X_eval)
    f1_rf, bacc_rf = evaluar_y_exportar(y_eval, yhat_rf, f'{profile} | RandomForest', f'{profile}_rf_{EVAL_ON}')

    pred_rf = M_eval.copy()
    pred_rf['y_pred'] = yhat_rf
    if hasattr(rf, 'predict_proba'):
        probs = rf.predict_proba(X_eval)
        for i, cls in enumerate(rf.classes_):
            pred_rf[f'prob_{cls}'] = probs[:, i]
    pred_rf.to_csv(OUT_DIR / f'predicciones_{profile}_rf_{EVAL_ON}.csv', index=False)

    filas.append({
        'profile': profile,
        'model': 'RF',
        'cv_macro_f1': rf_cv,
        'macro_f1': f1_rf,
        'balanced_acc': bacc_rf,
        'best_params': json.dumps(rf_params, ensure_ascii=False),
        'n_features': X_tr.shape[1],
        'n_train': len(X_tr),
        'n_eval': len(X_eval),
        'eval_split': EVAL_ON,
    })
    modelos['RF'] = rf

    # XGBoost
    if USE_XGB and HAS_XGB:
        xgb, xgb_params, xgb_cv, le_xgb = entrenar_xgb(X_tr, y_tr, groups=G_tr)
        yhat_raw = np.asarray(xgb.predict(X_eval))

        # Compatibilidad entre versiones de XGBoost:
        # algunas devuelven índices de clase (1D) y otras probabilidades (2D).
        if yhat_raw.ndim == 2:
            yhat_idx = np.argmax(yhat_raw, axis=1)
        else:
            yhat_idx = yhat_raw.astype(int)

        yhat_xgb = le_xgb.inverse_transform(np.asarray(yhat_idx).astype(int))
        f1_xgb, bacc_xgb = evaluar_y_exportar(y_eval, yhat_xgb, f'{profile} | XGBoost', f'{profile}_xgb_{EVAL_ON}')

        pred_xgb = M_eval.copy()
        pred_xgb['y_pred'] = yhat_xgb

        probs = None
        if hasattr(xgb, 'predict_proba'):
            probs = np.asarray(xgb.predict_proba(X_eval))
        elif yhat_raw.ndim == 2:
            probs = yhat_raw

        if probs is not None and probs.ndim == 2:
            n_cols = min(probs.shape[1], len(le_xgb.classes_))
            for i in range(n_cols):
                cls = le_xgb.classes_[i]
                pred_xgb[f'prob_{cls}'] = probs[:, i]

        pred_xgb.to_csv(OUT_DIR / f'predicciones_{profile}_xgb_{EVAL_ON}.csv', index=False)

        filas.append({
            'profile': profile,
            'model': 'XGB',
            'cv_macro_f1': xgb_cv,
            'macro_f1': f1_xgb,
            'balanced_acc': bacc_xgb,
            'best_params': json.dumps(xgb_params, ensure_ascii=False),
            'n_features': X_tr.shape[1],
            'n_train': len(X_tr),
            'n_eval': len(X_eval),
            'eval_split': EVAL_ON,
        })
        modelos['XGB'] = xgb
    else:
        if not USE_XGB:
            print('Aviso: Se omite XGBoost porque TRAIN_USE_XGB=0.')
        else:
            print('Aviso: Se omite XGBoost porque no está instalado en el entorno.')

    return filas, modelos


X_eval_core, y_eval_core, M_eval_core = (Xc_dev, yc_dev, Mc_dev) if EVAL_ON == 'dev' else (Xc_te, yc_te, Mc_te)
X_eval_py, y_eval_py, M_eval_py = (Xp_dev, yp_dev, Mp_dev) if EVAL_ON == 'dev' else (Xp_te, yp_te, Mp_te)

filas_core, modelos_core = ejecutar_experimentos('core', Xc_tr, yc_tr, Gc_tr, X_eval_core, y_eval_core, M_eval_core)
filas_py, modelos_py = ejecutar_experimentos('py', Xp_tr, yp_tr, Gp_tr, X_eval_py, y_eval_py, M_eval_py)

df_metricas = pd.DataFrame(filas_core + filas_py)
metricas_path = OUT_DIR / f'comparacion_modelos_{EVAL_ON}.csv'
df_metricas.to_csv(metricas_path, index=False)
print('Guardado:', metricas_path)

# Ablación por perfil léxico (`core` vs `py`)
abl = df_metricas.pivot_table(index='model', columns='profile', values='macro_f1', aggfunc='mean').reset_index()
if {'core', 'py'}.issubset(set(abl.columns)):
    abl['delta_py_vs_core'] = abl['py'] - abl['core']
abl_path = OUT_DIR / f'ablacion_perfiles_{EVAL_ON}.csv'
abl.to_csv(abl_path, index=False)
print('Guardado:', abl_path)

display(df_metricas.sort_values(['macro_f1', 'balanced_acc'], ascending=False))


Fitting 3 folds for each of 25 candidates, totalling 75 fits
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Guardado: /Users/manuelnunez/Projects/psych-phenotyping-paraguay/data/outputs/train_20260310_093418/comparacion_modelos_dev.csv
Guardado: /Users/manuelnunez/Projects/psych-phenotyping-paraguay/data/outputs/train_20260310_093418/ablacion_perfiles_dev.csv


,profile,model,cv_macro_f1,macro_f1,balanced_acc,best_params,n_features,n_train,n_eval,eval_split
3,py,XGB,NaN,0.688997,0.684753,"{""subsample"": 1.0, ""reg_lambda"": 1.0, ""n_estim...",958,1107,343,dev
1,core,XGB,NaN,0.681765,0.677695,"{""subsample"": 1.0, ""reg_lambda"": 1.0, ""n_estim...",954,1107,343,dev
0,core,RF,0.656522,0.677252,0.687716,"{""n_estimators"": 300, ""min_samples_split"": 2, ...",954,1107,343,dev
2,py,RF,0.659403,0.673086,0.682716,"{""n_estimators"": 300, ""min_samples_split"": 2, ...",958,1107,343,dev


In [7]:
# Guardado de modelos y columnas de características

def guardar_bundle(modelos: dict, x_cols: list[str], prefijo: str):
    for nombre, modelo in modelos.items():
        model_path = OUT_DIR / f'modelo_{prefijo}_{nombre}.joblib'
        joblib.dump(modelo, model_path)
        print('Modelo:', model_path.name)

    cols_path = OUT_DIR / f'{prefijo}_X_cols.json'
    with open(cols_path, 'w', encoding='utf-8') as f:
        json.dump(x_cols, f, ensure_ascii=False, indent=2)
    print('Columnas de features:', cols_path.name)


guardar_bundle(modelos_core, list(Xc_tr.columns), 'core')
guardar_bundle(modelos_py, list(Xp_tr.columns), 'py')

resumen = {
    'run_id': RUN_ID,
    'eval_on': EVAL_ON,
    'feature_run_id_core': FEATURE_RUN_ID_CORE,
    'feature_run_id_py': FEATURE_RUN_ID_PY,
    'use_randomized_search': USE_RANDOMIZED_SEARCH,
    'n_iter_search': N_ITER_SEARCH,
    'cv_folds': CV_FOLDS,
    'has_xgb': HAS_XGB,
    'use_xgb': USE_XGB,
    'require_xgb': REQUIRE_XGB,
}
with open(OUT_DIR / 'resumen_entrenamiento.json', 'w', encoding='utf-8') as f:
    json.dump(resumen, f, ensure_ascii=False, indent=2)

print('Entrenamiento híbrido finalizado.')
print('Carpeta de salida:', OUT_DIR)


Modelo: modelo_core_RF.joblib
Modelo: modelo_core_XGB.joblib
Columnas de features: core_X_cols.json
Modelo: modelo_py_RF.joblib
Modelo: modelo_py_XGB.joblib
Columnas de features: py_X_cols.json
Entrenamiento híbrido finalizado.
Carpeta de salida: /Users/manuelnunez/Projects/psych-phenotyping-paraguay/data/outputs/train_20260310_093418
